# 07 — Production governance & Spark deploy (Databricks)

This notebook outlines **Unity Catalog**, **job parameters**, and **governance** flows aligned with `examples/production/docs/`.

**Prerequisites:** DBR 14+ cluster, `pip install sparkrules[spark]` on the cluster, appropriate IAM for S3 / Glue.

## 1. Install pinned library

In [ ]:
%pip install 'sparkrules[spark]==1.1.0'

## 2. Load DRL from workspace / volume

In [ ]:
from pathlib import Path

drl_path = Path("/Workspace/Repos/your_team/policy/underwriting.drl")
drl = drl_path.read_text(encoding="utf-8") if drl_path.is_file() else '''
rule "demo" when $t : T ( $t.amount > 100 ) then result.tier = "gold"; end
'''
print(len(drl), "chars")

## 3. Score a Delta / Iceberg batch (V2)

Replace `INPUT_TABLE` with your catalog table. Facts must match DRL struct bindings.

In [ ]:
from pyspark.sql import functions as F
from sparkrules.spark import apply_drl

INPUT_TABLE = "main.raw.facts"  # TODO
df = spark.table(INPUT_TABLE)
# Example: wrap existing columns into struct `t` for pattern `$t : T`
if "t" not in df.columns:
    df = df.select(F.struct(F.col("amount").alias("amount")).alias("t"))
scored = apply_drl(df, drl, use_v2=True)
scored.write.format("delta").mode("overwrite").saveAsTable("main.curated.facts_scored")

## 4. Governance API (optional)

If your **SparkRules FastAPI** is reachable from the driver (private link), promote pins from a job or use a separate ops pipeline. Example with `requests`:

```python
import os, requests
base = os.environ["SPARKRULES_API_BASE"]  # https://rules.internal
h = {"X-Roles": "platform_admin", "X-Principal": "databricks-job", "X-Tenant-Id": "default"}
r = requests.post(f"{base}/governance/sync-dev", json={"namespace": "ns1", "rule_handle": "uw"}, headers=h, timeout=30)
r.raise_for_status()
```

See `examples/production/docs/GOVERNANCE_WORKFLOW.md`.

## 5. Next steps

- Wire **Kafka → Iceberg** streaming: `examples/streaming/kafka_iceberg_structured_streaming.py`
- Import **Grafana** dashboard: `examples/production/grafana/grafana-sparkrules.json`
- Run **cluster benchmark** protocol: `examples/production/benchmarks/BENCHMARK_CLUSTER.md`